<a href="https://colab.research.google.com/github/akemitti/Pred-inad-credito/blob/main/notebook06.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Notebook 06 — Integração dos Sentimentos com a Previsão de Inadimplência


1. carregar a base ampla exportada pelo Notebook 05: `base_completa.csv`;
2. identificar automaticamente todos os scores de sentimento disponíveis;
3. testar, para cada tipo de relatório e para cada modelo de sentimento, os lags de 1 a 6 meses;
4. estimar modelos **ARIMAX** — modelo ARIMA com variáveis exógenas, isto é, uma série temporal com variáveis explicativas externas — para previsão da inadimplência em horizonte H=3;
5. comparar os resultados por **RMSE** — raiz do erro quadrático médio, métrica que penaliza erros maiores —, além de MAE e R².

A comparação é feita com a mesma base, o mesmo período, o mesmo split de treino/teste e o mesmo alvo `target_h3` para evitar comparação injusta entre modelos.


## 1. Instalação e importações


In [1]:
%pip install pandas numpy matplotlib seaborn statsmodels scikit-learn scipy --quiet



[notice] A new release of pip is available: 25.3 -> 26.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import re
import warnings
import unicodedata
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.tools.sm_exceptions import ConvergenceWarning

warnings.filterwarnings("ignore")
warnings.simplefilter("ignore", ConvergenceWarning)

SEED = 42
np.random.seed(SEED)

# Configuração visual
COLOR_BASE = "#aec7e8"
COLOR_SENT = "#0e5764"
COLOR_REAL = "#d62728"
COLOR_SPLIT = "gray"

plt.rcParams.update({
    "figure.dpi": 130,
    "font.size": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

print("Bibliotecas carregadas.")


Bibliotecas carregadas.


## 2. Parâmetros do experimento

O notebook usa previsão direta para H=3. Isso significa que, em cada mês `t`, o modelo tenta prever a inadimplência três meses à frente, isto é, `inad_total` em `t+3`.


In [3]:
# Horizonte de previsão
H = 3

# Mesmo split utilizado no Notebook 02, se este foi o corte aprovado na etapa anterior.
# Ajuste aqui caso o Notebook 02 tenha sido consolidado com outro corte temporal.
SPLIT_DATE = "2023-01-01"
SPLIT_TS = pd.to_datetime(SPLIT_DATE)

# Ordem ARIMA usada dentro do ARIMAX.
# Use a ordem vencedora/mais estável do Notebook 02. Pelo histórico do projeto, (2, 0, 0) foi usada como referência.
ARIMA_ORDER = (2, 0, 0)
MAXITER = 200

# Lags autorregressivos da inadimplência usados como base explicativa.
LAG_FEATS = [f"inad_total_lag{i}" for i in range(1, 7)]

# Modelos de sentimento esperados na base do Notebook 05.
MODELOS_ESPERADOS = ["nltk", "textblob", "bert", "finbert", "mistral", "media"]
TIPOS_RELATORIO_ESPERADOS = ["copom", "estatisticas"]
LAGS_SENTIMENTO = list(range(1, 7))

print(f"Horizonte H={H}")
print(f"Split treino/teste: treino < {SPLIT_TS.date()} | teste >= {SPLIT_TS.date()}")
print(f"Ordem ARIMA no ARIMAX: {ARIMA_ORDER}")


Horizonte H=3
Split treino/teste: treino < 2023-01-01 | teste >= 2023-01-01
Ordem ARIMA no ARIMAX: (2, 0, 0)


## 3. Funções auxiliares


In [4]:
def normalizar_coluna(coluna: str) -> str:
    """Padroniza nomes de colunas para evitar erro com maiúsculas, acentos e espaços."""
    coluna = str(coluna).strip().lower()
    coluna = unicodedata.normalize("NFKD", coluna).encode("ascii", "ignore").decode("ascii")
    coluna = coluna.replace(" ", "_").replace("-", "_")
    coluna = re.sub(r"__+", "_", coluna)
    return coluna


def metricas_regr(y_true, y_pred) -> dict:
    """Calcula MAE, RMSE, R², bias e reta previsto~observado."""
    y_true = pd.Series(y_true).astype(float)
    y_pred = pd.Series(y_pred).astype(float)

    df_tmp = pd.concat([y_true.rename("obs"), y_pred.rename("pred")], axis=1).dropna()
    if len(df_tmp) == 0:
        return {"MAE": np.nan, "RMSE": np.nan, "R2": np.nan, "Bias (obs-prev)": np.nan,
                "Slope pred~obs": np.nan, "Intercept": np.nan}

    mae = mean_absolute_error(df_tmp["obs"], df_tmp["pred"])
    rmse = np.sqrt(mean_squared_error(df_tmp["obs"], df_tmp["pred"]))
    r2 = r2_score(df_tmp["obs"], df_tmp["pred"]) if len(df_tmp) >= 2 else np.nan
    bias = float(np.mean(df_tmp["obs"] - df_tmp["pred"]))

    if len(df_tmp) >= 2:
        lr = LinearRegression().fit(df_tmp[["obs"]], df_tmp["pred"])
        slope = float(lr.coef_[0])
        intercept = float(lr.intercept_)
    else:
        slope = np.nan
        intercept = np.nan

    return {
        "MAE": round(float(mae), 6),
        "RMSE": round(float(rmse), 6),
        "R2": round(float(r2), 6) if pd.notna(r2) else np.nan,
        "Bias (obs-prev)": round(float(bias), 6),
        "Slope pred~obs": round(float(slope), 6) if pd.notna(slope) else np.nan,
        "Intercept": round(float(intercept), 6) if pd.notna(intercept) else np.nan,
    }


def detectar_colunas_sentimento(df: pd.DataFrame) -> pd.DataFrame:
    """
    Identifica colunas no padrão:
    tipo_score_modelo_lag
    Exemplos:
    copom_score_mistral_l1, estatisticas_score_finbert_l3.
    """
    padrao = re.compile(r"^(copom|estatisticas)_score_(.+)_l([1-6])$")
    registros = []

    for col in df.columns:
        match = padrao.match(col)
        if match:
            tipo, modelo, lag = match.groups()
            modelo = modelo.replace("media_dos_modelos", "media").replace("media_modelos", "media")
            registros.append({
                "tipo_relatorio": tipo,
                "modelo_sentimento": modelo,
                "lag": int(lag),
                "coluna": col,
            })

    out = pd.DataFrame(registros)
    if not out.empty:
        out = out.sort_values(["tipo_relatorio", "modelo_sentimento", "lag"]).reset_index(drop=True)
    return out


def garantir_lags_inadimplencia(df: pd.DataFrame, alvo="inad_total") -> pd.DataFrame:
    """Cria inad_total_lag1..6 caso a base ainda não contenha essas colunas."""
    df = df.copy()
    for lag in range(1, 7):
        col = f"{alvo}_lag{lag}"
        if col not in df.columns:
            df[col] = df[alvo].shift(lag)
    return df


def rodar_arimax_walk_forward(
    nome: str,
    df_full: pd.DataFrame,
    features: list,
    split_ts: pd.Timestamp,
    target_col: str = "target_h3",
    data_col: str = "data",
    order: tuple = ARIMA_ORDER,
    maxiter: int = MAXITER,
) -> dict:
    """
    Roda ARIMAX em walk-forward expansivo.

    Walk-forward expansivo = reestima o modelo a cada nova observação do teste,
    sempre usando apenas informações disponíveis até aquele ponto.
    """
    cols = [data_col, target_col] + features
    df_m = df_full[cols].dropna().sort_values(data_col).reset_index(drop=True)

    train_size = int((df_m[data_col] < split_ts).sum())
    test_size = len(df_m) - train_size

    if train_size < 12:
        raise ValueError(f"[{nome}] treino insuficiente: {train_size} observações.")
    if test_size < 1:
        raise ValueError(f"[{nome}] teste vazio. Verifique SPLIT_DATE.")

    preds = []
    idxs = []
    erros = 0

    for i in range(train_size, len(df_m)):
        y_train = df_m.loc[:i-1, target_col].astype(float)
        X_train = df_m.loc[:i-1, features].astype(float)
        X_test = df_m.loc[[i], features].astype(float)

        try:
            modelo = SARIMAX(
                endog=y_train,
                exog=X_train,
                order=order,
                trend="c",
                enforce_stationarity=False,
                enforce_invertibility=False,
            )
            ajuste = modelo.fit(disp=False, maxiter=maxiter)
            pred = float(ajuste.get_forecast(steps=1, exog=X_test).predicted_mean.iloc[0])
        except Exception:
            pred = np.nan
            erros += 1

        preds.append(pred)
        idxs.append(i)

    yhat = pd.Series(preds, index=idxs, name="yhat")
    y_true = df_m.loc[idxs, target_col].astype(float).rename("y_true")
    datas = df_m.loc[idxs, data_col]

    validos = yhat.dropna().index
    yhat = yhat.loc[validos]
    y_true = y_true.loc[validos]
    datas = datas.loc[validos]

    bias = float((y_true - yhat).mean()) if len(yhat) else np.nan
    yhat_bc = yhat + bias if pd.notna(bias) else yhat

    return {
        "nome": nome,
        "features": features,
        "datas": datas,
        "y_true": y_true,
        "yhat": yhat,
        "yhat_bc": yhat_bc,
        "bias": bias,
        "metricas": metricas_regr(y_true, yhat),
        "metricas_bc": metricas_regr(y_true, yhat_bc),
        "train_size": train_size,
        "test_size": test_size,
        "n_predicoes_validas": len(yhat),
        "n_erros_ajuste": erros,
    }

print("Funções auxiliares definidas.")


Funções auxiliares definidas.


## 4. Carregamento da base do Notebook 05

O Notebook 06 passa a usar **somente** a base ampla exportada pelo Notebook 05:

`base_sentimento_lags_todos_relatorios.csv`

Essa base já deve conter a série de inadimplência, os lags da inadimplência e os lags de sentimento de 1 a 6 para Copom e Estatísticas.


In [5]:
ARQUIVO_BASE = "base_completa.csv"
URL_BASE = "https://raw.githubusercontent.com/akemitti/Pred-inad-credito/main/base_completa.csv"

try:
    df = pd.read_csv(ARQUIVO_BASE)
    print(f"Base carregada do disco: {ARQUIVO_BASE}")
except FileNotFoundError:
    df = pd.read_csv(URL_BASE)
    print("Base carregada do GitHub.")

# Padronizar nomes das colunas.
df.columns = [normalizar_coluna(c) for c in df.columns]

# Padronizar coluna de data.
if "data" in df.columns:
    df["data"] = pd.to_datetime(df["data"], errors="coerce")
elif "mes" in df.columns:
    df["data"] = pd.to_datetime(df["mes"], errors="coerce")
else:
    raise KeyError(f"A base precisa ter uma coluna de data ou mes. Colunas disponíveis: {list(df.columns)}")

# Garantir frequência mensal no primeiro dia do mês.
df["data"] = df["data"].dt.to_period("M").dt.to_timestamp()
df = df.sort_values("data").reset_index(drop=True)

# Recorte temporal consolidado no projeto.
df = df[df["data"] <= "2025-12-31"].reset_index(drop=True)

print(f"Período da base: {df['data'].min().date()} → {df['data'].max().date()}")
print(f"Linhas: {len(df)} | Colunas: {len(df.columns)}")
display(df.head())


Base carregada do disco: base_completa.csv
Período da base: 2019-07-01 → 2025-12-01
Linhas: 78 | Colunas: 81


,unnamed:_0,data,inad_total,inad_total_lag1,inad_total_lag2,inad_total_lag3,inad_total_lag4,inad_total_lag5,inad_total_lag6,copom_score_nltk_l1,...,estatisticas_score_mistral_l3,estatisticas_score_mistral_l4,estatisticas_score_mistral_l5,estatisticas_score_mistral_l6,estatisticas_score_media_l1,estatisticas_score_media_l2,estatisticas_score_media_l3,estatisticas_score_media_l4,estatisticas_score_media_l5,estatisticas_score_media_l6
0,0,2019-07-01,3.06,2.95,3.05,3.02,2.99,2.91,2.95,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,2019-08-01,3.04,3.06,2.95,3.05,3.02,2.99,2.91,-0.84020,...,NaN,NaN,NaN,NaN,-0.359629,NaN,NaN,NaN,NaN,NaN
2,2,2019-09-01,3.06,3.04,3.06,2.95,3.05,3.02,2.99,-0.86045,...,NaN,NaN,NaN,NaN,-0.489411,-0.359629,NaN,NaN,NaN,NaN
3,3,2019-10-01,3.03,3.06,3.04,3.06,2.95,3.05,3.02,-0.88070,...,-0.1,NaN,NaN,NaN,-0.341160,-0.489411,-0.359629,NaN,NaN,NaN
4,4,2019-11-01,3.00,3.03,3.06,3.04,3.06,2.95,3.05,-0.84020,...,-0.2,-0.1,NaN,NaN,-0.385238,-0.341160,-0.489411,-0.359629,NaN,NaN


## 5. Validação da estrutura da base

Aqui o notebook verifica se estão disponíveis os seis lags de cada modelo de sentimento. A base esperada contém modelos como NLTK, TextBlob, BERT, FinBERT, Mistral e Média dos Modelos para os relatórios `copom` e `estatisticas`.


In [6]:
# Garantir que a inadimplência está presente.
if "inad_total" not in df.columns:
    raise KeyError("A coluna 'inad_total' não foi encontrada na base.")

# Criar lags de inadimplência caso eles não existam.
df = garantir_lags_inadimplencia(df, alvo="inad_total")

# Criar target H=3.
df["target_h3"] = df["inad_total"].shift(-H)

# Identificar colunas de sentimento.
df_sent_cols = detectar_colunas_sentimento(df)

if df_sent_cols.empty:
    raise ValueError(
        "Nenhuma coluna de sentimento foi encontrada. "
        "O padrão esperado é, por exemplo, 'copom_score_mistral_L1' ou 'estatisticas_score_finbert_L3'."
    )

print("Colunas de sentimento detectadas:")
display(df_sent_cols)

# Checagem dos lags esperados por tipo/modelo.
linhas_check = []
for tipo in sorted(df_sent_cols["tipo_relatorio"].unique()):
    for modelo in sorted(df_sent_cols.loc[df_sent_cols["tipo_relatorio"] == tipo, "modelo_sentimento"].unique()):
        lags_disp = sorted(df_sent_cols.query("tipo_relatorio == @tipo and modelo_sentimento == @modelo")["lag"].tolist())
        linhas_check.append({
            "tipo_relatorio": tipo,
            "modelo_sentimento": modelo,
            "lags_disponiveis": lags_disp,
            "qtd_lags": len(lags_disp),
            "tem_L1_a_L6": lags_disp == [1, 2, 3, 4, 5, 6],
        })

df_check_lags = pd.DataFrame(linhas_check)
display(df_check_lags)

faltantes = df_check_lags[df_check_lags["tem_L1_a_L6"] == False]
if len(faltantes) > 0:
    print("Atenção: há modelos sem todos os lags de L1 a L6. Eles serão testados apenas com os lags disponíveis.")
else:
    print("Todos os modelos detectados possuem L1 a L6.")


Colunas de sentimento detectadas:


,tipo_relatorio,modelo_sentimento,lag,coluna
0,copom,bert,1,copom_score_bert_l1
1,copom,bert,2,copom_score_bert_l2
2,copom,bert,3,copom_score_bert_l3
3,copom,bert,4,copom_score_bert_l4
4,copom,bert,5,copom_score_bert_l5
...,...,...,...,...
67,estatisticas,textblob,2,estatisticas_score_textblob_l2
68,estatisticas,textblob,3,estatisticas_score_textblob_l3
69,estatisticas,textblob,4,estatisticas_score_textblob_l4
70,estatisticas,textblob,5,estatisticas_score_textblob_l5


,tipo_relatorio,modelo_sentimento,lags_disponiveis,qtd_lags,tem_L1_a_L6
0,copom,bert,"[1, 2, 3, 4, 5, 6]",6,True
1,copom,finbert,"[1, 2, 3, 4, 5, 6]",6,True
2,copom,media,"[1, 2, 3, 4, 5, 6]",6,True
3,copom,mistral,"[1, 2, 3, 4, 5, 6]",6,True
4,copom,nltk,"[1, 2, 3, 4, 5, 6]",6,True
5,copom,textblob,"[1, 2, 3, 4, 5, 6]",6,True
6,estatisticas,bert,"[1, 2, 3, 4, 5, 6]",6,True
7,estatisticas,finbert,"[1, 2, 3, 4, 5, 6]",6,True
8,estatisticas,media,"[1, 2, 3, 4, 5, 6]",6,True
9,estatisticas,mistral,"[1, 2, 3, 4, 5, 6]",6,True


Todos os modelos detectados possuem L1 a L6.


## 6. Construção da base de modelagem

Para evitar comparação injusta, o notebook cria uma base única de avaliação. Isso garante que todos os modelos sejam comparados no mesmo período, com o mesmo target e com o mesmo split de treino/teste.


In [7]:
SENTIMENTO_COLS = df_sent_cols["coluna"].tolist()
COLS_OBRIGATORIAS = ["data", "inad_total", "target_h3"] + LAG_FEATS + SENTIMENTO_COLS

# Usar apenas colunas que existem, por segurança.
COLS_OBRIGATORIAS = [c for c in COLS_OBRIGATORIAS if c in df.columns]

n_antes = len(df)
df_model = df[COLS_OBRIGATORIAS].dropna().sort_values("data").reset_index(drop=True)
n_depois = len(df_model)

print(f"Linhas antes do dropna: {n_antes}")
print(f"Linhas após base comum de modelagem: {n_depois}")
print(f"Período de modelagem: {df_model['data'].min().date()} → {df_model['data'].max().date()}")
print(f"Treino: {(df_model['data'] < SPLIT_TS).sum()} obs | Teste: {(df_model['data'] >= SPLIT_TS).sum()} obs")

nulos = df_model[COLS_OBRIGATORIAS].isna().sum()
if nulos.sum() == 0:
    print("Nenhum NaN nas colunas de avaliação. Comparação justa entre modelos.")
else:
    display(nulos[nulos > 0])

display(df_model.head())


Linhas antes do dropna: 78
Linhas após base comum de modelagem: 69
Período de modelagem: 2020-01-01 → 2025-09-01
Treino: 36 obs | Teste: 33 obs
Nenhum NaN nas colunas de avaliação. Comparação justa entre modelos.


,data,inad_total,target_h3,inad_total_lag1,inad_total_lag2,inad_total_lag3,inad_total_lag4,inad_total_lag5,inad_total_lag6,copom_score_bert_l1,...,estatisticas_score_nltk_l3,estatisticas_score_nltk_l4,estatisticas_score_nltk_l5,estatisticas_score_nltk_l6,estatisticas_score_textblob_l1,estatisticas_score_textblob_l2,estatisticas_score_textblob_l3,estatisticas_score_textblob_l4,estatisticas_score_textblob_l5,estatisticas_score_textblob_l6
0,2020-01-01,3.00,3.29,2.94,3.00,3.03,3.06,3.04,3.06,-0.250000,...,-0.9929,-0.9911,-0.9911,-0.9921,-0.320000,0.000000,0.140000,-0.228571,-0.320000,-0.133333
1,2020-02-01,3.04,3.24,3.00,2.94,3.00,3.03,3.06,3.04,-0.208333,...,-0.9884,-0.9929,-0.9911,-0.9911,-0.114286,-0.320000,0.000000,0.140000,-0.228571,-0.320000
2,2020-03-01,3.18,2.89,3.04,3.00,2.94,3.00,3.03,3.06,-0.166667,...,-0.9911,-0.9884,-0.9929,-0.9911,0.000000,-0.114286,-0.320000,0.000000,0.140000,-0.228571
3,2020-04-01,3.29,2.76,3.18,3.04,3.00,2.94,3.00,3.03,-0.166667,...,-0.9974,-0.9911,-0.9884,-0.9929,0.000000,0.000000,-0.114286,-0.320000,0.000000,0.140000
4,2020-05-01,3.24,2.65,3.29,3.18,3.04,3.00,2.94,3.00,-0.291667,...,-0.9904,-0.9974,-0.9911,-0.9884,0.000000,0.000000,0.000000,-0.114286,-0.320000,0.000000


## 7. Modelo base ARIMAX sem sentimento

O modelo base usa apenas os lags da inadimplência como variáveis explicativas. Ele será a referência para medir se os sentimentos melhoram ou pioram a previsão.


In [8]:
print("=" * 80)
print("MODELO BASE — ARIMAX sem sentimento")
print("=" * 80)

res_base = rodar_arimax_walk_forward(
    nome="ARIMAX Base",
    df_full=df_model,
    features=LAG_FEATS,
    split_ts=SPLIT_TS,
    order=ARIMA_ORDER,
)

print(f"Treino: {res_base['train_size']} | Teste: {res_base['test_size']} | Predições válidas: {res_base['n_predicoes_validas']}")
print(f"Erros de ajuste: {res_base['n_erros_ajuste']}")

print("\nMétricas sem correção de viés:")
display(pd.DataFrame([res_base["metricas"]], index=["ARIMAX Base"]))

print(f"Métricas com correção de viés ex-post (bias={res_base['bias']:+.6f}):")
display(pd.DataFrame([res_base["metricas_bc"]], index=["ARIMAX Base + bias"]))


MODELO BASE — ARIMAX sem sentimento


Treino: 36 | Teste: 33 | Predições válidas: 33
Erros de ajuste: 0

Métricas sem correção de viés:


,MAE,RMSE,R2,Bias (obs-prev),Slope pred~obs,Intercept
ARIMAX Base,0.078472,0.105801,0.86355,0.028882,0.946634,0.154437


Métricas com correção de viés ex-post (bias=+0.028882):


,MAE,RMSE,R2,Bias (obs-prev),Slope pred~obs,Intercept
ARIMAX Base + bias,0.079474,0.101783,0.873718,-0.0,0.946634,0.183319


## 8. ARIMAX + sentimentos: teste de todos os modelos com L1 a L6

Nesta etapa, cada coluna de sentimento é testada separadamente. Assim, por exemplo, o Mistral é avaliado como:

- ARIMAX + `copom_score_mistral_l1`
- ARIMAX + `copom_score_mistral_l2`
- ARIMAX + `copom_score_mistral_l3`
- ARIMAX + `copom_score_mistral_l4`
- ARIMAX + `copom_score_mistral_l5`
- ARIMAX + `copom_score_mistral_l6`

A mesma lógica é aplicada para NLTK, TextBlob, BERT, FinBERT e Média dos Modelos, tanto para Copom quanto para Estatísticas, quando as colunas estiverem disponíveis.


In [ ]:
resultados = []
resultados_detalhados = {}

rmse_base = res_base["metricas_bc"]["RMSE"]
mae_base = res_base["metricas_bc"]["MAE"]
r2_base = res_base["metricas_bc"]["R2"]

total_modelos = len(df_sent_cols)
print(f"Total de combinações sentimento-lag a testar: {total_modelos}")

for i, row in df_sent_cols.iterrows():
    tipo = row["tipo_relatorio"]
    modelo_sent = row["modelo_sentimento"]
    lag = int(row["lag"])
    col_sent = row["coluna"]

    nome = f"ARIMAX + {tipo} | {modelo_sent} | L{lag}"
    print(f"[{i+1:02d}/{total_modelos:02d}] {nome}")

    try:
        res = rodar_arimax_walk_forward(
            nome=nome,
            df_full=df_model,
            features=LAG_FEATS + [col_sent],
            split_ts=SPLIT_TS,
            order=ARIMA_ORDER,
        )

        m = res["metricas_bc"]
        delta_rmse_pct = (rmse_base - m["RMSE"]) / rmse_base * 100
        delta_mae_pct = (mae_base - m["MAE"]) / mae_base * 100
        delta_r2 = m["R2"] - r2_base

        resultados.append({
            "modelo": nome,
            "tipo_relatorio": tipo,
            "modelo_sentimento": modelo_sent,
            "lag": lag,
            "coluna_sentimento": col_sent,
            "ordem_arima": str(ARIMA_ORDER),
            "n_treino": res["train_size"],
            "n_teste": res["test_size"],
            "n_predicoes_validas": res["n_predicoes_validas"],
            "n_erros_ajuste": res["n_erros_ajuste"],
            "MAE": m["MAE"],
            "RMSE": m["RMSE"],
            "R2": m["R2"],
            "Bias (obs-prev)": m["Bias (obs-prev)"],
            "Delta_RMSE_pct_vs_base": round(delta_rmse_pct, 4),
            "Delta_MAE_pct_vs_base": round(delta_mae_pct, 4),
            "Delta_R2_vs_base": round(delta_r2, 6),
        })

        resultados_detalhados[nome] = res

    except Exception as e:
        resultados.append({
            "modelo": nome,
            "tipo_relatorio": tipo,
            "modelo_sentimento": modelo_sent,
            "lag": lag,
            "coluna_sentimento": col_sent,
            "ordem_arima": str(ARIMA_ORDER),
            "n_treino": np.nan,
            "n_teste": np.nan,
            "n_predicoes_validas": np.nan,
            "n_erros_ajuste": np.nan,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "Bias (obs-prev)": np.nan,
            "Delta_RMSE_pct_vs_base": np.nan,
            "Delta_MAE_pct_vs_base": np.nan,
            "Delta_R2_vs_base": np.nan,
            "erro": str(e),
        })
        print(f"  Erro: {e}")


df_resultados = pd.DataFrame(resultados)
df_resultados = df_resultados.sort_values("RMSE", na_position="last").reset_index(drop=True)

print("\nRanking geral por menor RMSE:")
display(df_resultados)


df_resultados


Total de combinações sentimento-lag a testar: 72
[01/72] ARIMAX + copom | bert | L1


[02/72] ARIMAX + copom | bert | L2
[03/72] ARIMAX + copom | bert | L3
[04/72] ARIMAX + copom | bert | L4
[05/72] ARIMAX + copom | bert | L5
[06/72] ARIMAX + copom | bert | L6
[07/72] ARIMAX + copom | finbert | L1
[08/72] ARIMAX + copom | finbert | L2
[09/72] ARIMAX + copom | finbert | L3
[10/72] ARIMAX + copom | finbert | L4
[11/72] ARIMAX + copom | finbert | L5
[12/72] ARIMAX + copom | finbert | L6
[13/72] ARIMAX + copom | media | L1
[14/72] ARIMAX + copom | media | L2
[15/72] ARIMAX + copom | media | L3
[16/72] ARIMAX + copom | media | L4
[17/72] ARIMAX + copom | media | L5
[18/72] ARIMAX + copom | media | L6
[19/72] ARIMAX + copom | mistral | L1
[20/72] ARIMAX + copom | mistral | L2
[21/72] ARIMAX + copom | mistral | L3
[22/72] ARIMAX + copom | mistral | L4
[23/72] ARIMAX + copom | mistral | L5
[24/72] ARIMAX + copom | mistral | L6
[25/72] ARIMAX + copom | nltk | L1
[26/72] ARIMAX + copom | nltk | L2
[27/72] ARIMAX + copom | nltk | L3
[28/72] ARIMAX + copom | nltk | L4
[29/72] ARIMA

## 9. Análise por modelo de sentimento: L1 a L6

A célula abaixo organiza os resultados de forma parecida com o pedido do professor: para cada modelo de sentimento, aparecem os seis lags avaliados em ARIMAX.


In [ ]:
def exibir_analise_por_modelo(df_resultados: pd.DataFrame, modelo_sentimento: str):
    """Exibe a tabela L1-L6 de um modelo de sentimento específico."""
    base = df_resultados[df_resultados["modelo_sentimento"] == modelo_sentimento].copy()
    if base.empty:
        print(f"Modelo '{modelo_sentimento}' não encontrado na base de resultados.")
        return

    base = base.sort_values(["tipo_relatorio", "lag"])
    cols = [
        "tipo_relatorio", "modelo_sentimento", "lag", "coluna_sentimento",
        "RMSE", "MAE", "R2", "Delta_RMSE_pct_vs_base", "n_predicoes_validas", "n_erros_ajuste"
    ]
    print("=" * 100)
    print(f"ANÁLISE DO MODELO DE SENTIMENTO: {modelo_sentimento.upper()} — ARIMAX com L1 a L6")
    print("=" * 100)
    display(base[cols])

    melhor = base.dropna(subset=["RMSE"]).sort_values("RMSE").head(1)
    if len(melhor) > 0:
        linha = melhor.iloc[0]
        print(
            f"Melhor configuração para {modelo_sentimento}: "
            f"{linha['tipo_relatorio']} L{int(linha['lag'])} | "
            f"RMSE={linha['RMSE']:.6f} | "
            f"ganho vs base={linha['Delta_RMSE_pct_vs_base']:+.2f}%"
        )

for modelo in sorted(df_resultados["modelo_sentimento"].dropna().unique()):
    exibir_analise_por_modelo(df_resultados, modelo)


## 10. Melhor lag de cada modelo de sentimento

Esta tabela resume qual lag teve melhor desempenho preditivo para cada combinação de tipo de relatório e modelo de sentimento.


In [ ]:
df_melhor_por_modelo = (
    df_resultados
    .dropna(subset=["RMSE"])
    .sort_values("RMSE")
    .groupby(["tipo_relatorio", "modelo_sentimento"], as_index=False)
    .first()
    .sort_values("RMSE")
)

cols_resumo = [
    "tipo_relatorio", "modelo_sentimento", "lag", "coluna_sentimento",
    "RMSE", "MAE", "R2", "Delta_RMSE_pct_vs_base", "Delta_MAE_pct_vs_base", "Delta_R2_vs_base"
]

print("Melhor lag por tipo de relatório e modelo de sentimento:")
display(df_melhor_por_modelo[cols_resumo])

df_melhor_por_modelo.to_csv("melhor_lag_por_modelo_sentimento_notebook06.csv", index=False)
print("Arquivo exportado: melhor_lag_por_modelo_sentimento_notebook06.csv")


## 11. Comparativo final: modelo base vs. melhores modelos com sentimento

Nesta etapa, o notebook seleciona o menor RMSE de cada modelo de sentimento e compara com o ARIMAX base.


In [ ]:
linha_base = {
    "tipo_relatorio": "base",
    "modelo_sentimento": "sem_sentimento",
    "lag": "-",
    "coluna_sentimento": "-",
    "RMSE": res_base["metricas_bc"]["RMSE"],
    "MAE": res_base["metricas_bc"]["MAE"],
    "R2": res_base["metricas_bc"]["R2"],
    "Delta_RMSE_pct_vs_base": 0.0,
    "Delta_MAE_pct_vs_base": 0.0,
    "Delta_R2_vs_base": 0.0,
}

df_comp_final = pd.concat([
    pd.DataFrame([linha_base]),
    df_melhor_por_modelo[cols_resumo]
], ignore_index=True).sort_values("RMSE").reset_index(drop=True)

print("Comparativo final — base vs melhores configurações de sentimento:")
display(df_comp_final)

melhor_geral = df_comp_final.iloc[0]
print(
    "\nMelhor resultado geral pelo menor RMSE: "
    f"{melhor_geral['tipo_relatorio']} | {melhor_geral['modelo_sentimento']} | "
    f"L{melhor_geral['lag']} | RMSE={melhor_geral['RMSE']:.6f}"
)

df_comp_final.to_csv("comparativo_final_notebook06_arimax.csv", index=False)
print("Arquivo exportado: comparativo_final_notebook06_arimax.csv")


## 12. Visualizações dos resultados

Os gráficos abaixo ajudam a enxergar quais lags melhoraram ou pioraram o desempenho em relação ao modelo base.


In [ ]:
# Heatmap de RMSE por tipo de relatório.
for tipo in sorted(df_resultados["tipo_relatorio"].dropna().unique()):
    dados_tipo = df_resultados[df_resultados["tipo_relatorio"] == tipo].copy()
    if dados_tipo.empty:
        continue

    pivot_rmse = dados_tipo.pivot_table(
        index="modelo_sentimento",
        columns="lag",
        values="RMSE",
        aggfunc="min",
    )

    plt.figure(figsize=(9, 4.8))
    sns.heatmap(pivot_rmse, annot=True, fmt=".4f", cmap="Blues", linewidths=0.5)
    plt.title(f"RMSE por modelo de sentimento e lag — {tipo.upper()}", fontweight="bold")
    plt.xlabel("Lag do sentimento")
    plt.ylabel("Modelo de sentimento")
    plt.tight_layout()
    plt.show()


In [ ]:
# Gráfico de ganho/perda de RMSE vs. modelo base.
df_plot = df_resultados.dropna(subset=["Delta_RMSE_pct_vs_base"]).copy()
df_plot["config"] = (
    df_plot["tipo_relatorio"] + " | " +
    df_plot["modelo_sentimento"] + " | L" +
    df_plot["lag"].astype(int).astype(str)
)

df_plot_top = df_plot.sort_values("Delta_RMSE_pct_vs_base", ascending=False).head(20)

plt.figure(figsize=(12, 6))
plt.barh(df_plot_top["config"], df_plot_top["Delta_RMSE_pct_vs_base"])
plt.axvline(0, color="black", linewidth=1)
plt.gca().invert_yaxis()
plt.title("Top 20 ganhos de RMSE em relação ao ARIMAX base", fontweight="bold")
plt.xlabel("Ganho de RMSE vs. base (%)")
plt.ylabel("Configuração")
plt.tight_layout()
plt.show()


## 13. Série temporal: observado vs. base vs. melhor modelo com sentimento

Este gráfico compara a inadimplência observada, a previsão do ARIMAX base e a previsão do melhor modelo com sentimento.


In [ ]:
# Identificar melhor modelo com sentimento, excluindo a linha base.
df_melhor_sent = df_resultados.dropna(subset=["RMSE"]).sort_values("RMSE").head(1)

if len(df_melhor_sent) > 0:
    nome_melhor = df_melhor_sent.iloc[0]["modelo"]
    res_melhor = resultados_detalhados[nome_melhor]

    fig, ax = plt.subplots(figsize=(15, 5))

    ax.plot(
        res_base["datas"].values,
        res_base["y_true"].values,
        color=COLOR_REAL,
        lw=2.4,
        label="Inadimplência observada",
        zorder=5,
    )

    ax.plot(
        res_base["datas"].values,
        res_base["yhat_bc"].values,
        color=COLOR_BASE,
        lw=1.8,
        linestyle="--",
        label=f"ARIMAX Base | RMSE={res_base['metricas_bc']['RMSE']:.4f}",
    )

    ax.plot(
        res_melhor["datas"].values,
        res_melhor["yhat_bc"].values,
        color=COLOR_SENT,
        lw=2.2,
        label=f"{nome_melhor} | RMSE={res_melhor['metricas_bc']['RMSE']:.4f}",
    )

    ax.axvline(SPLIT_TS, color=COLOR_SPLIT, linestyle=":", lw=1.3, label="Split treino/teste")
    ax.set_title("Previsão H=3 — ARIMAX Base vs. Melhor ARIMAX com Sentimento", fontweight="bold")
    ax.set_xlabel("Mês")
    ax.set_ylabel("Inadimplência total (%)")
    ax.legend(frameon=False, fontsize=9)
    ax.grid(True, alpha=0.25)
    plt.tight_layout()
    plt.show()
else:
    print("Não foi possível identificar melhor modelo com sentimento.")


## 14. Interpretação metodológica para o TCC

A etapa preditiva do Notebook 06 desloca a seleção final das variáveis textuais para o desempenho dos modelos de previsão. Em vez de escolher os sentimentos apenas pela correlação com a inadimplência, cada modelo de sentimento é testado com defasagens de 1 a 6 meses em uma estrutura ARIMAX. A variável dependente é a inadimplência três meses à frente (`target_h3`), enquanto as variáveis explicativas incluem os lags da própria inadimplência e, em cada especificação, uma variável de sentimento defasada.

A comparação entre os modelos é realizada por RMSE, MAE e R², com o RMSE como critério principal. Para garantir comparabilidade, todos os modelos são avaliados sobre a mesma janela temporal, com o mesmo conjunto de teste e o mesmo horizonte de previsão.

Os arquivos exportados por este notebook são:

- `resultados_notebook06_arimax_sentimentos_lags.csv`: ranking completo de todos os modelos, tipos de relatório e lags;
- `melhor_lag_por_modelo_sentimento_notebook06.csv`: melhor lag de cada modelo de sentimento;
- `comparativo_final_notebook06_arimax.csv`: comparação entre o modelo base e os melhores modelos com sentimento.
